In [2]:
import h5py
import numpy as np
import os
import sys
import time
import multiprocessing
from functools import partial
from tqdm import tqdm

# Importa suas ferramentas
sys.path.insert(0, os.path.abspath(".."))
from tools import utils


In [3]:
# ==============================================================================
# CONFIGURATION SECTION
# ==============================================================================

# 1. SIMULATION INFO
# Select '025' for TNG50-4 or '264' for MN5
SNAP_NUM = "025"          

# Output Label (matches the folder name in notebooks/mini_sims_library_{NAME})
SIM_NAME = "TNG50-4"      

# Original Simulation Path (Source)
BASE_DATA_PATH = "/home/STORAGE/TNG50-4" 

# 2. SELECTION MODE
# True: Use Francisco's Xmatch list (MN5 style). 
# False: Process ALL halos in the box (TNG style).
USE_XMATCH = False 

# 3. OUTPUT DESTINATION
# Dynamic path based on SIM_NAME
# Ideally, keep this structure to allow run_parallel.py to find it easily
# Path Handling for Jupyter vs Terminal
try:
    PROJ_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    PROJ_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

LIBRARY_OUTPUT_DIR = os.path.join(PROJ_ROOT, f"notebooks/mini_sims_library_{SIM_NAME}")
XMATCH_DIR = os.path.join(PROJ_ROOT, "xmatchs")

# 4. BOX GEOMETRY (FIXED CUBIC BOX)
# Defines the size of the mini-simulation box in Mpc/h.
# The code will automatically convert this to kpc/h if the source is TNG.
# 2.0 Mpc is safe for storage (~150GB total) and covers the R200 of most halos.
FIXED_BOX_MPC = 2.0 

# 5. PARALLELISM
NUM_WORKERS = 28  # Number of cores

print(f"--> Project Root detected: {PROJ_ROOT}")
print(f"--> Output Library:        {LIBRARY_OUTPUT_DIR}")

# ==============================================================================

--> Project Root detected: /home/vroberto/FakeSpectra_DLAs
--> Output Library:        /home/vroberto/FakeSpectra_DLAs/notebooks/mini_sims_library_TNG50-4


In [4]:

def smart_read(dataset, mask):
    """
    Reads dataset with fallback to uint64 if it is uint48 (<u6).
    Essential for TNG ID reads preventing TypeError.
    """
    try:
        # Try direct read
        data = dataset[:]
        return data[mask]
    except TypeError:
        # Fallback for custom numpy dtypes (like uint48 in some TNG files)
        shape = dataset.shape
        buffer = np.empty(shape, dtype=np.uint64)
        dataset.read_direct(buffer)
        return buffer[mask]

def process_single_halo(task_index, output_root):
    """
    Worker Function: Extracts a cubic region around a halo.
    Handles unit conversion (Mpc <-> kpc) automatically.
    Filters out invalid halos (R200=0).
    """
    try:
        # A. INITIALIZE CONTEXT
        # Re-instantiate inside the worker to prevent HDF5 handle conflicts
        ctx = utils.SimulationContext(BASE_DATA_PATH, SNAP_NUM)
        halo_finder = utils.HaloFinder(ctx)
        
        # B. DETERMINE TARGET HALO ID
        if USE_XMATCH:
            matcher = utils.HaloMatcher(XMATCH_DIR)
            real_halo_id = matcher.get_halo_id(SIM_NAME, task_index)
            halo_lookup_id = real_halo_id
        else:
            # Full Box Mode: The task_index IS the Halo ID
            real_halo_id = task_index
            halo_lookup_id = task_index

        # C. GET HALO PROPERTIES
        # This returns 'cm' (Center of Mass) in NATIVE units (kpc or Mpc)
        try:
            halo_props = halo_finder.get_halo_properties(halo_lookup_id)
        except ValueError:
             return f"[SKIP] Halo {real_halo_id} not found/out of bounds."

        # --- [CRITICAL] FILTER INVALID HALOS ---
        # If R200 is zero or Mass is zero, this halo is a FOF artifact.
        # Skipping it prevents division by zero errors in the analysis steps.
        r200_val = halo_props.get('r200_native', 0.0)
        mass_val = halo_props.get('mass', 0.0)

        if r200_val <= 1e-6 or mass_val <= 0:
            return f"[SKIP] Halo {real_halo_id} is invalid (R200={r200_val:.2e})"
        # ---------------------------------------

        # D. PREPARE OUTPUT DIRECTORIES
        halo_dir_name = f"halo_{real_halo_id}"
        current_output_dir = os.path.join(output_root, halo_dir_name)
        
        # Check if already done (Resume capability)
        final_file = os.path.join(current_output_dir, f"snapdir_{ctx.snap_num}", f"snapshot_{ctx.snap_num}.0.hdf5")
        if os.path.exists(final_file):
            return f"[SKIP] Halo {real_halo_id} already exists."

        # E. GEOMETRY & UNITS LOGIC
        # We want a fixed box of FIXED_BOX_MPC.
        # If simulation is in kpc, we must scale the cut-out box.
        
        native_box_size = halo_props['box_size'] # Simulation full box
        halo_center_native = halo_props['cm']
        
        if ctx.units == 'kpc':
            target_box_native = FIXED_BOX_MPC * 1000.0
        else:
            target_box_native = FIXED_BOX_MPC
            
        half_box_native = target_box_native / 2.0
        
        # The new center in the NEW sub-box will be exactly in the middle
        new_center_pos = np.array([half_box_native, half_box_native, half_box_native])

        # Create Folders
        snap_dir_name = f"snapdir_{ctx.snap_num}"
        group_dir_name = f"groups_{ctx.snap_num}"
        
        os.makedirs(os.path.join(current_output_dir, snap_dir_name), exist_ok=True)
        os.makedirs(os.path.join(current_output_dir, group_dir_name), exist_ok=True)

        # --- PART 1: SNAPSHOT EXTRACTION ---
        out_snap_file = os.path.join(current_output_dir, snap_dir_name, f"snapshot_{ctx.snap_num}.0.hdf5")
        first_snap = ctx.get_snap_file(0)
        
        with h5py.File(first_snap, 'r') as f_in, h5py.File(out_snap_file, 'w') as f_out:
            # 1. Copy Header
            header = f_out.create_group("Header")
            for k, v in f_in['Header'].attrs.items(): header.attrs[k] = v
            
            # OVERWRITE BOX SIZE with the Mini-Sim size (in native units)
            header.attrs['BoxSize'] = target_box_native
            header.attrs['NumFilesPerSnapshot'] = 1
            
            # 2. Handle 'Parameters' (Essential for MN5 compatibility tools)
            # TNG often lacks 'Parameters', so we create it if missing to help fake_spectra
            if "Parameters" in f_in:
                f_in.copy("Parameters", f_out)
                # Ensure BoxSize in Parameters matches
                if "BoxSize" in f_out["Parameters"].attrs:
                    f_out["Parameters"].attrs["BoxSize"] = target_box_native
            else:
                # Create dummy Parameters for TNG
                params = f_out.create_group("Parameters")
                # Copy relevant physics attrs from Header to Parameters if they exist
                for k in ['HubbleParam', 'Omega0', 'OmegaLambda', 'UnitLength_in_cm', 'UnitMass_in_g', 'UnitVelocity_in_cm_per_s']:
                    if k in header.attrs: params.attrs[k] = header.attrs[k]
                params.attrs["BoxSize"] = target_box_native

            # 3. Particle Loop
            n_files_orig = int(f_in['Header'].attrs['NumFilesPerSnapshot'])
            total_parts_by_type = np.zeros(6, dtype=np.int64)

            for pt_idx in range(6):
                pt_name = f"PartType{pt_idx}"
                accumulated_data = {}
                has_data = False
                
                # Iterate over ALL files (Linear scan is safest for TNG structure)
                for i in range(n_files_orig):
                    try:
                        f_chunk = h5py.File(ctx.get_snap_file(i), 'r')
                    except: continue
                    
                    if pt_name not in f_chunk: 
                        f_chunk.close(); continue

                    coords = f_chunk[pt_name]['Coordinates'][:]
                    
                    # Calculate distance from Halo Center (Native Units)
                    dx = coords - halo_center_native
                    
                    # Apply PBC (using Original Box Size)
                    dx[dx > native_box_size/2.0] -= native_box_size
                    dx[dx < -native_box_size/2.0] += native_box_size
                    
                    # Mask based on Target Box Size
                    mask = (np.abs(dx[:, 0]) <= half_box_native) & \
                           (np.abs(dx[:, 1]) <= half_box_native) & \
                           (np.abs(dx[:, 2]) <= half_box_native)
                    
                    if np.sum(mask) > 0:
                        has_data = True
                        # Shift coordinates to the new local center
                        new_coords = dx[mask] + new_center_pos
                        
                        for key in f_chunk[pt_name].keys():
                            if key not in accumulated_data: accumulated_data[key] = []
                            
                            if key == 'Coordinates': 
                                accumulated_data[key].append(new_coords)
                            else: 
                                dset = f_chunk[pt_name][key]
                                accumulated_data[key].append(smart_read(dset, mask))
                    
                    f_chunk.close()
                
                # Write accumulated data for this Particle Type
                if has_data:
                    g_out = f_out.create_group(pt_name)
                    for key, data_list in accumulated_data.items():
                        full_data = np.concatenate(data_list, axis=0)
                        g_out.create_dataset(key, data=full_data, compression="gzip")
                        if key == 'Coordinates': total_parts_by_type[pt_idx] = full_data.shape[0]

            # Update Counts in Header
            header.attrs['NumPart_ThisFile'] = total_parts_by_type
            header.attrs['NumPart_Total'] = total_parts_by_type
            header.attrs['NumPart_Total_HighWord'] = np.zeros(6, dtype=np.int64)

        # --- PART 2: GROUP FILE CREATION ---
        # We create a fake group file containing only the central halo
        # This allows fake_spectra to "load" the halo from the mini-sim easily.
        
        out_group_file = os.path.join(current_output_dir, group_dir_name, f"fof_subhalo_tab_{ctx.snap_num}.0.hdf5")
        
        # We need to open a TNG group file to copy structure (File 0 is safe)
        with h5py.File(ctx.get_group_file(0), 'r') as g_in, h5py.File(out_group_file, 'w') as g_out:
            # Header
            header = g_out.create_group("Header")
            for k, v in g_in['Header'].attrs.items(): header.attrs[k] = v
            
            header.attrs['BoxSize'] = target_box_native
            header.attrs['Ngroups_Total'] = 1
            header.attrs['Ngroups_ThisFile'] = 1
            header.attrs['NumFiles'] = 1
            
            if "Parameters" in g_in: g_in.copy("Parameters", g_out)

            # Group Data
            grp = g_out.create_group("Group")
            
            # Position: Placed exactly at the center of the new box
            grp.create_dataset("GroupPos", data=np.array([new_center_pos]))
            
            # Mass & R200: Copied from original properties
            grp.create_dataset("Group_M_Crit200", data=np.array([halo_props['mass']]))
            # Note: R200 is written in native units (fake_spectra expects native)
            grp.create_dataset("Group_R_Crit200", data=np.array([halo_props['r200_native']]))
            
            # Velocity: Convert physical back to code units for consistency
            scale_factor = header.attrs['Time']
            # vel_pec = raw_vel * sqrt(a)  -> raw_vel = vel_pec / sqrt(a)
            raw_vel_native = halo_props['vel_pec'] / np.sqrt(scale_factor)
            grp.create_dataset("GroupVel", data=np.array([raw_vel_native]))

        return f"[OK] Halo {real_halo_id}. Box: {target_box_native:.1f} {ctx.units}"

    except Exception as e:
        return f"[ERROR] Task {task_index}: {e}"

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
def main():
    print("==================================================")
    print(f"   MINI-SIMULATION GENERATOR (Universal)")
    print(f"   Simulation : {SIM_NAME}")
    print(f"   Path       : {BASE_DATA_PATH}")
    print(f"   Mode       : {'XMATCH' if USE_XMATCH else 'FULL BOX'}")
    print(f"   Output     : {LIBRARY_OUTPUT_DIR}")
    print(f"   Box Target : {FIXED_BOX_MPC} Mpc/h")
    print(f"   Workers    : {NUM_WORKERS}")
    print("==================================================\n")
    
    os.makedirs(LIBRARY_OUTPUT_DIR, exist_ok=True)
    
    # 1. Determine Task List
    if USE_XMATCH:
        try:
            matcher = utils.HaloMatcher(XMATCH_DIR)
            cat_len = len(matcher.match_catalogs.get(SIM_NAME, []))
            # Trigger load if empty
            if cat_len == 0: 
                 matcher.get_halo_id(SIM_NAME, 0)
                 cat_len = len(matcher.match_catalogs[SIM_NAME])
            ALL_TASKS = list(range(cat_len))
            print(f"--> Found {len(ALL_TASKS)} halos in Xmatch catalog.")
        except Exception as e:
            print(f"[CRITICAL] Xmatch setup failed: {e}")
            sys.exit(1)
    else:
        try:
            temp_ctx = utils.SimulationContext(BASE_DATA_PATH, SNAP_NUM)
            total_halos = temp_ctx.get_total_groups()
            ALL_TASKS = list(range(total_halos))
            print(f"--> Found {len(ALL_TASKS)} total halos in simulation header.")
        except Exception as e:
            print(f"[CRITICAL] Failed to read simulation header: {e}")
            sys.exit(1)

    # 2. Run Parallel
    start_time = time.time()
    worker_func = partial(process_single_halo, output_root=LIBRARY_OUTPUT_DIR)
    
    with multiprocessing.Pool(processes=NUM_WORKERS) as pool:
        # Using imap_unordered for better performance with progress bar
        results = list(tqdm(pool.imap_unordered(worker_func, ALL_TASKS), total=len(ALL_TASKS), unit="halo"))
    
    # 3. Report
    print("\n=== FINAL REPORT ===")
    errors = [r for r in results if "[ERROR]" in r]
    success = [r for r in results if "[OK]" in r]
    skips = [r for r in results if "[SKIP]" in r]
    
    print(f"Success : {len(success)}")
    print(f"Skipped : {len(skips)} (Invalid Halos or Already Done)")
    print(f"Errors  : {len(errors)}")
    
    if errors:
        print("\n--- ERROR SAMPLES ---")
        for err in errors[:5]: print(err)
        if len(errors) > 5: print("...")

    print(f"\nTotal Time: {(time.time() - start_time)/60:.2f} min")

In [5]:
if __name__ == "__main__":
    main()

2026-03-02 15:29:29,685 - INFO - Detected Simulation Units: kpc


   MINI-SIMULATION GENERATOR (Universal)
   Simulation : TNG50-4
   Path       : /home/STORAGE/TNG50-4
   Mode       : FULL BOX
   Output     : /home/vroberto/FakeSpectra_DLAs/notebooks/mini_sims_library_TNG50-4
   Box Target : 2.0 Mpc/h
   Workers    : 28

--> Found 29158 total halos in simulation header.


  0%|          | 0/29158 [00:00<?, ?halo/s]2026-03-02 15:29:29,772 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,772 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,772 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,772 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,772 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,772 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,772 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,774 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,774 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,774 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,774 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,775 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,775 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,776 - INFO - Detected Simulation Units: kpc
2026-03-02 15:29:29,777 - INFO - Detected Simulation Units: k


=== FINAL REPORT ===
Success : 0
Skipped : 3619 (Invalid Halos or Already Done)
Errors  : 25539

--- ERROR SAMPLES ---
[ERROR] Task 23: "Unable to synchronously open object (object 'Coordinates' doesn't exist)"
[ERROR] Task 24: "Unable to synchronously open object (object 'Coordinates' doesn't exist)"
[ERROR] Task 25: "Unable to synchronously open object (object 'Coordinates' doesn't exist)"
[ERROR] Task 13: "Unable to synchronously open object (object 'Coordinates' doesn't exist)"
[ERROR] Task 21: "Unable to synchronously open object (object 'Coordinates' doesn't exist)"
...

Total Time: 376.85 min


# TESTE, APAGAR


In [7]:
import h5py
import numpy as np
import os
import sys
import time
import multiprocessing
from functools import partial
from tqdm import tqdm

# Importa suas ferramentas
sys.path.insert(0, os.path.abspath(".."))
from tools import utils

# ==============================================================================
# CONFIGURATION SECTION
# ==============================================================================

# 1. SIMULATION INFO
SNAP_NUM = "025"          # "025" for TNG, "264" for MN5
SIM_NAME = "TNG50-4"      # Output Label
BASE_DATA_PATH = "/home/STORAGE/TNG50-4" # Source Path

# 2. SELECTION MODE
# False: Process ALL halos in the box (TNG style).
# True: Use Xmatch list (MN5 style).
USE_XMATCH = False 

# 3. OUTPUT DESTINATION
try:
    PROJ_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    PROJ_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

LIBRARY_OUTPUT_DIR = os.path.join(PROJ_ROOT, f"notebooks/mini_sims_library_{SIM_NAME}")
XMATCH_DIR = os.path.join(PROJ_ROOT, "xmatchs")

# 4. BOX GEOMETRY
# 2.0 Mpc is safe and sufficient for R200 analysis.
FIXED_BOX_MPC = 2 

# 5. PARALLELISM
NUM_WORKERS = 25  

print(f"--> Project Root detected: {PROJ_ROOT}")
print(f"--> Output Library:        {LIBRARY_OUTPUT_DIR}")

# ==============================================================================

def smart_read(dataset, mask):
    """
    Reads dataset with fallback to uint64 if it is uint48 (<u6).
    """
    try:
        data = dataset[:]
        return data[mask]
    except TypeError:
        shape = dataset.shape
        buffer = np.empty(shape, dtype=np.uint64)
        dataset.read_direct(buffer)
        return buffer[mask]

def process_single_halo(task_index, output_root):
    """
    Worker Function: Extracts a cubic region around a halo.
    Includes robust checks for missing datasets in chunks.
    """
    try:
        # A. INITIALIZE CONTEXT
        ctx = utils.SimulationContext(BASE_DATA_PATH, SNAP_NUM)
        halo_finder = utils.HaloFinder(ctx)
        
        # B. DETERMINE TARGET HALO ID
        if USE_XMATCH:
            matcher = utils.HaloMatcher(XMATCH_DIR)
            real_halo_id = matcher.get_halo_id(SIM_NAME, task_index)
            halo_lookup_id = real_halo_id
        else:
            real_halo_id = task_index
            halo_lookup_id = task_index

        # C. GET HALO PROPERTIES
        try:
            halo_props = halo_finder.get_halo_properties(halo_lookup_id)
        except ValueError:
             return f"[SKIP] Halo {real_halo_id} not found."

        # Filter Invalid Halos (R200=0 or Mass=0)
        r200_val = halo_props.get('r200_native', 0.0)
        mass_val = halo_props.get('mass', 0.0)

        if r200_val <= 1e-6 or mass_val <= 0:
            return f"[SKIP] Halo {real_halo_id} invalid (R200=0)."

        # D. PREPARE OUTPUT DIRECTORIES
        halo_dir_name = f"halo_{real_halo_id}"
        current_output_dir = os.path.join(output_root, halo_dir_name)
        
        # Check if complete (resume capability)
        # We check the group file existence as a marker of success
        final_file = os.path.join(current_output_dir, f"groups_{ctx.snap_num}", f"fof_subhalo_tab_{ctx.snap_num}.0.hdf5")
        if os.path.exists(final_file):
            return f"[SKIP] Halo {real_halo_id} already exists."

        # E. GEOMETRY & UNITS LOGIC
        native_box_size = halo_props['box_size']
        halo_center_native = halo_props['cm']
        
        if ctx.units == 'kpc':
            target_box_native = FIXED_BOX_MPC * 1000.0
        else:
            target_box_native = FIXED_BOX_MPC
            
        half_box_native = target_box_native / 2.0
        new_center_pos = np.array([half_box_native, half_box_native, half_box_native])

        snap_dir_name = f"snapdir_{ctx.snap_num}"
        group_dir_name = f"groups_{ctx.snap_num}"
        
        os.makedirs(os.path.join(current_output_dir, snap_dir_name), exist_ok=True)
        os.makedirs(os.path.join(current_output_dir, group_dir_name), exist_ok=True)

        # --- PART 1: SNAPSHOT EXTRACTION ---
        out_snap_file = os.path.join(current_output_dir, snap_dir_name, f"snapshot_{ctx.snap_num}.0.hdf5")
        first_snap = ctx.get_snap_file(0)
        
        with h5py.File(first_snap, 'r') as f_in, h5py.File(out_snap_file, 'w') as f_out:
            # Header
            header = f_out.create_group("Header")
            for k, v in f_in['Header'].attrs.items(): header.attrs[k] = v
            header.attrs['BoxSize'] = target_box_native
            header.attrs['NumFilesPerSnapshot'] = 1
            
            # Parameters (Create/Copy)
            if "Parameters" in f_in:
                f_in.copy("Parameters", f_out)
                if "BoxSize" in f_out["Parameters"].attrs:
                    f_out["Parameters"].attrs["BoxSize"] = target_box_native
            else:
                params = f_out.create_group("Parameters")
                # Copy key physics params
                for k in ['HubbleParam', 'Omega0', 'OmegaLambda', 'UnitLength_in_cm', 'UnitMass_in_g', 'UnitVelocity_in_cm_per_s']:
                    if k in header.attrs: params.attrs[k] = header.attrs[k]
                params.attrs["BoxSize"] = target_box_native

            # Particles Loop
            n_files_orig = int(f_in['Header'].attrs['NumFilesPerSnapshot'])
            total_parts_by_type = np.zeros(6, dtype=np.int64)

            for pt_idx in range(6):
                pt_name = f"PartType{pt_idx}"
                accumulated_data = {}
                has_data = False
                
                for i in range(n_files_orig):
                    try:
                        f_chunk = h5py.File(ctx.get_snap_file(i), 'r')
                    except: continue
                    
                    # --- ROBUST DATASET CHECK ---
                    if pt_name not in f_chunk: 
                        f_chunk.close(); continue
                    
                    # CRITICAL FIX: Check if 'Coordinates' exists before accessing
                    if 'Coordinates' not in f_chunk[pt_name]:
                        f_chunk.close(); continue
                        
                    try:
                        coords = f_chunk[pt_name]['Coordinates'][:]
                    except Exception:
                        # Skip corrupt chunk
                        f_chunk.close(); continue
                    # ----------------------------

                    dx = coords - halo_center_native
                    dx[dx > native_box_size/2.0] -= native_box_size
                    dx[dx < -native_box_size/2.0] += native_box_size
                    
                    mask = (np.abs(dx[:, 0]) <= half_box_native) & \
                           (np.abs(dx[:, 1]) <= half_box_native) & \
                           (np.abs(dx[:, 2]) <= half_box_native)
                    
                    if np.sum(mask) > 0:
                        has_data = True
                        new_coords = dx[mask] + new_center_pos
                        
                        for key in f_chunk[pt_name].keys():
                            if key not in accumulated_data: accumulated_data[key] = []
                            
                            if key == 'Coordinates': 
                                accumulated_data[key].append(new_coords)
                            else: 
                                # Check key existence to be safe
                                if key in f_chunk[pt_name]:
                                    dset = f_chunk[pt_name][key]
                                    accumulated_data[key].append(smart_read(dset, mask))
                    
                    f_chunk.close()
                
                if has_data:
                    g_out = f_out.create_group(pt_name)
                    for key, data_list in accumulated_data.items():
                        full_data = np.concatenate(data_list, axis=0)
                        g_out.create_dataset(key, data=full_data, compression="gzip")
                        if key == 'Coordinates': total_parts_by_type[pt_idx] = full_data.shape[0]

            header.attrs['NumPart_ThisFile'] = total_parts_by_type
            header.attrs['NumPart_Total'] = total_parts_by_type
            header.attrs['NumPart_Total_HighWord'] = np.zeros(6, dtype=np.int64)

        # --- PART 2: GROUP FILE CREATION ---
        out_group_file = os.path.join(current_output_dir, group_dir_name, f"fof_subhalo_tab_{ctx.snap_num}.0.hdf5")
        
        with h5py.File(ctx.get_group_file(0), 'r') as g_in, h5py.File(out_group_file, 'w') as g_out:
            header = g_out.create_group("Header")
            for k, v in g_in['Header'].attrs.items(): header.attrs[k] = v
            
            header.attrs['BoxSize'] = target_box_native
            header.attrs['Ngroups_Total'] = 1
            header.attrs['Ngroups_ThisFile'] = 1
            header.attrs['NumFiles'] = 1
            
            if "Parameters" in g_in: g_in.copy("Parameters", g_out)

            grp = g_out.create_group("Group")
            grp.create_dataset("GroupPos", data=np.array([new_center_pos]))
            grp.create_dataset("Group_M_Crit200", data=np.array([halo_props['mass']]))
            grp.create_dataset("Group_R_Crit200", data=np.array([halo_props['r200_native']]))
            
            scale_factor = header.attrs['Time']
            raw_vel_native = halo_props['vel_pec'] / np.sqrt(scale_factor)
            grp.create_dataset("GroupVel", data=np.array([raw_vel_native]))

        return f"[OK] Halo {real_halo_id}"

    except Exception as e:
        return f"[ERROR] Task {task_index}: {e}"

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
def main():
    print("==================================================")
    print(f"   MINI-SIM GENERATOR (FIXED)")
    print(f"   Sim: {SIM_NAME} | Path: {BASE_DATA_PATH}")
    print(f"   Output: {LIBRARY_OUTPUT_DIR}")
    print("==================================================\n")
    
    os.makedirs(LIBRARY_OUTPUT_DIR, exist_ok=True)
    
    if USE_XMATCH:
        try:
            matcher = utils.HaloMatcher(XMATCH_DIR)
            cat_len = len(matcher.match_catalogs.get(SIM_NAME, []))
            if cat_len == 0: 
                 matcher.get_halo_id(SIM_NAME, 0)
                 cat_len = len(matcher.match_catalogs[SIM_NAME])
            ALL_TASKS = list(range(cat_len))
            print(f"--> Found {len(ALL_TASKS)} halos in Xmatch.")
        except Exception as e:
            print(f"[CRITICAL] Xmatch setup failed: {e}")
            sys.exit(1)
    else:
        try:
            temp_ctx = utils.SimulationContext(BASE_DATA_PATH, SNAP_NUM)
            total_halos = temp_ctx.get_total_groups()
            ALL_TASKS = list(range(total_halos))
            print(f"--> Found {len(ALL_TASKS)} total halos.")
        except Exception as e:
            print(f"[CRITICAL] Failed to read simulation header: {e}")
            sys.exit(1)

    # --- TEST MODE ---
    # Uncomment next line to test 1 halo before full run
    # ALL_TASKS = [0] 

    start_time = time.time()
    worker_func = partial(process_single_halo, output_root=LIBRARY_OUTPUT_DIR)
    
    with multiprocessing.Pool(processes=NUM_WORKERS) as pool:
        results = list(tqdm(pool.imap_unordered(worker_func, ALL_TASKS), total=len(ALL_TASKS), unit="halo"))
    
    # Report
    print("\n=== FINAL REPORT ===")
    errors = [r for r in results if "[ERROR]" in r]
    success = [r for r in results if "[OK]" in r]
    skips = [r for r in results if "[SKIP]" in r]
    
    print(f"Success : {len(success)}")
    print(f"Skipped : {len(skips)}")
    print(f"Errors  : {len(errors)}")
    
    if errors:
        print("\n--- ERROR SAMPLES ---")
        for err in errors[:5]: print(err)

    print(f"\nTotal Time: {(time.time() - start_time)/60:.2f} min")

if __name__ == "__main__":
    main()

2026-03-03 13:28:12,855 - INFO - Detected Simulation Units: kpc


--> Project Root detected: /home/vroberto/FakeSpectra_DLAs
--> Output Library:        /home/vroberto/FakeSpectra_DLAs/notebooks/mini_sims_library_TNG50-4
   MINI-SIM GENERATOR (FIXED)
   Sim: TNG50-4 | Path: /home/STORAGE/TNG50-4
   Output: /home/vroberto/FakeSpectra_DLAs/notebooks/mini_sims_library_TNG50-4

--> Found 29158 total halos.


2026-03-03 13:28:13,814 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,815 - INFO - Detected Simulation Units: kpc
  0%|          | 0/29158 [00:00<?, ?halo/s]2026-03-03 13:28:13,817 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,836 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,830 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,833 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,852 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,857 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,847 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,861 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,857 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,854 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,864 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,836 - INFO - Detected Simulation Units: kpc
2026-03-03 13:28:13,864 - INFO - Detected Simulation Units: k


=== FINAL REPORT ===
Success : 25539
Skipped : 3619
Errors  : 0

Total Time: 439.37 min
